# nested_conditions

Nested conditions appear whenever a system makes layered decisions: authenticate, then authorize, then validate state, then execute. They are sometimes necessary, but unmanaged nesting quickly becomes a reliability problem because humans lose track of why a branch was taken.


## 1. Problem First

Why does this matter in real systems?

- A deployment action may require environment validity, approval, and artifact integrity.
- An API endpoint may check identity, permissions, rate limits, and payload validity.
- Deep nesting hides which condition actually blocked the request.


In [1]:
user_authenticated = True
user_is_admin = True
request_is_safe = False

if user_authenticated:
    if user_is_admin:
        if request_is_safe:
            decision = "allow"
        else:
            decision = "deny_unsafe_request"
    else:
        decision = "deny_insufficient_role"
else:
    decision = "deny_unauthenticated"

print(decision)

deny_unsafe_request


## 2. Minimal Concept

Core syntax:

- You can place an `if` statement inside another `if` block.
- Inner blocks only run when outer conditions already passed.
- Nesting expresses dependency between checks.


## 3. Mental Model

How Python actually behaves

Nested conditions create a path tree. Each level narrows the execution path further. The deeper the nesting, the more context a reader must hold in memory to understand why a line runs. The runtime handles this easily; humans often do not.


In [2]:
request = {
    "authenticated": True,
    "has_token": True,
    "quota_ok": False,
}

if request["authenticated"]:
    if request["has_token"]:
        if request["quota_ok"]:
            print("process request")
        else:
            print("quota exceeded")

quota exceeded


## 4. Internal Mechanics

Nested conditions compile into multiple conditional jumps. There is nothing special about the nesting itself; it is just layered branching. What matters is whether the structure reflects business rules clearly or buries them.


In [3]:
import dis

def authorize(user, resource_locked):
    if user["active"]:
        if user["is_admin"]:
            if not resource_locked:
                return "allow"
    return "deny"

dis.dis(authorize)
print(authorize({"active": True, "is_admin": True}, False))

  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (user)
              4 LOAD_CONST               1 ('active')
              6 BINARY_SUBSCR
             10 POP_JUMP_IF_FALSE        8 (to 28)

  5          12 LOAD_FAST                0 (user)
             14 LOAD_CONST               2 ('is_admin')
             16 BINARY_SUBSCR
             20 POP_JUMP_IF_FALSE        3 (to 28)

  6          22 LOAD_FAST                1 (resource_locked)
             24 POP_JUMP_IF_TRUE         1 (to 28)

  7          26 RETURN_CONST             3 ('allow')

  8     >>   28 RETURN_CONST             4 ('deny')
allow


## 5. Edge Cases

Where it breaks:

- Deep nesting hides the exact failure reason.
- Repeated dictionary access or complex expressions make branches noisy.
- Missing `else` paths often lead to implicit `None` or incomplete decisions.
- Falsey values can accidentally block execution when the real issue is different.


In [4]:
def validate(config):
    if config:
        if config.get("port"):
            return "valid"
    return "invalid"

print(validate({"port": 0}))  # 0 may be valid, but branch fails

invalid


## 6. Performance Thinking

- Branch checks are usually O(1), but nested trees increase path complexity.
- The real cost is maintainability and debugging time.
- Guard clauses often reduce both nesting depth and cognitive load.
- Expensive inner checks should only happen after cheap outer filters when that order is correct.


## 7. Real Use Case

A deployment tool should only release if the environment is allowed, the artifact exists, and manual approval is present.


In [5]:
environment = "prod"
artifact_exists = True
approved = True

if environment == "prod":
    if artifact_exists:
        if approved:
            result = "deploy"
        else:
            result = "wait_for_approval"
    else:
        result = "missing_artifact"
else:
    result = "invalid_environment"

print(result)

deploy


## 8. Anti-Pattern

What beginners do wrong:

- Keep nesting instead of using early returns or guard clauses.
- Mix validation, authorization, and side effects in the same nested block.
- Return a single vague failure after many nested checks.


In [6]:
def bad_process(user, payload):
    if user:
        if user.get("active"):
            if payload:
                if payload.get("id"):
                    return "process"
    return "fail"

print(bad_process({"active": True}, {"id": 0}))

fail


## 9. Interview Signals

Questions FAANG asks:

- When is nesting justified versus harmful?
- How do guard clauses improve nested logic?
- What are the risks of returning one generic error after many checks?
- How would you refactor a 4-level nested branch into something easier to test?


## 10. Exercise (Non-trivial)

You are reviewing an authorization function that checks user existence, account state, role, region restrictions, and resource locks. Refactor the nested version into guard clauses or clearer decision stages, and ensure each failure returns a precise reason.


In [7]:
def can_access(user, region_allowed, resource_locked):
    if user:
        if user.get("active"):
            if user.get("role") == "admin":
                if region_allowed:
                    if not resource_locked:
                        return "allow"
    return "deny"

# Task:
# 1. Refactor to reduce nesting.
# 2. Return specific failure reasons.
# 3. Explain why the new version is easier to test and debug.